# 10 IEEE 2890-2025 informed provenance review

**Deliverables:** Linked provenance catalog, file integrity manifest, lineage issue report, and full-standard review worksheet. Build evidence that a reviewer can trace.

**Scenario:** Synthetic water-monitoring data for fictional Example River Nation. Roles and decisions are classroom examples.

**Objectives:** Explain the framework, implement a documented workflow, inspect a failed case, and revise the result. Each lab runs independently; lessons 00–06 provide background. Allow 60–90 minutes.

No real authorization, Label issuance, certification, or technical access enforcement results from running these cells.

In [1]:
LESSON = '10_ieee_provenance_review'
import sys, json, csv
from pathlib import Path
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT/'toolkit'/'__init__.py').exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise FileNotFoundError('Run from inside IDSov_toolkit')
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from toolkit.teaching import load_context, output_directory
# Applied labs explicitly use fictional data, even when nation.yaml exists.
ctx = load_context(REPO_ROOT, 'config/teaching_example.yaml')
out = output_directory(REPO_ROOT, ctx) / LESSON
out.mkdir(parents=True, exist_ok=True)
def save_json(name, obj):
    (out/name).write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding='utf-8')
def save_csv(name, rows):
    with (out/name).open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
print('Fictional teaching scenario:', ctx.config.name)
print('Draft artifacts:', out)


Fictional teaching scenario: Example River Nation (fictional)
Draft artifacts: C:\Users\gekek\Documents\IDSov_toolkit\outputs\teaching\10_ieee_provenance_review


## Separate local implementation from standard requirements

IEEE 2890-2025 is the Recommended Practice for Provenance of Indigenous Peoples’ Data. This lab uses the toolkit schema and a conceptual mapping; the full standard has not been assessed clause by clause. No official clause identifiers or conformance claims are invented. A reviewer with access to the standard must populate the requirement worksheet. A hash verifies bytes, not authority or truth.


In [2]:
from toolkit.provenance import ProvenanceBuilder, ProvenanceCatalog
import hashlib
source = (ProvenanceBuilder('lab10-source-v1').for_nation(ctx).for_dataset('Invented water measurements').collected_by('Fictional classroom team', method='Synthetic fixture', date='2024-01').classify('RESTRICTED').note('Invented values; no actual collection or consent.').build())
derived = (ProvenanceBuilder('lab10-summary-v1').for_nation(ctx).for_dataset('Synthetic water summary').processed_by('Fictional analyst', purpose='Classroom seasonal planning').derived_from(source.record_id).step('Arithmetic mean of two invented values in metres: 2.4 and 2.1').classify('RESTRICTED').note('Local fictional datum; no environmental inference.').build())
catalog = ProvenanceCatalog('Fictional provenance review')
for record in [source, derived]: catalog.add(record)
catalog.save(out/'provenance_catalog.json')
assert derived.publication_status == 'pending'


Provenance catalog saved: C:\Users\gekek\Documents\IDSov_toolkit\outputs\teaching\10_ieee_provenance_review\provenance_catalog.json


## Inspect broken lineage before packaging

The catalog permits duplicate IDs and unresolved parents, so review them explicitly here. This local check also detects cycles. It does not verify source authenticity or the adequacy of a governance decision.


In [3]:
def lineage_issues(records):
    ids = [r['record_id'] for r in records]
    issues = ['Duplicate ID: '+i for i in sorted(set(ids)) if ids.count(i)>1]
    graph = {r['record_id']:r.get('derived_from',[]) for r in records}
    for record in records:
        issues.extend('Missing parent: '+p+' for '+record['record_id'] for p in record.get('derived_from',[]) if p not in graph)
    active, done = set(), set()
    def visit(node):
        if node in active:
            issues.append('Cycle at: '+node)
            return
        if node in done or node not in graph: return
        active.add(node)
        for parent in graph[node]: visit(parent)
        active.remove(node)
        done.add(node)
    for node in graph: visit(node)
    return issues
records = catalog.to_dict()['records']
assert lineage_issues(records) == []
broken = [dict(records[0], derived_from=['missing-source']), dict(records[1], derived_from=[records[1]['record_id']]), records[0]]
issues = lineage_issues(broken)
assert any('Duplicate' in i for i in issues)
assert any('Missing parent' in i for i in issues)
assert any('Cycle' in i for i in issues)
save_json('lineage_review.json', {'valid_catalog_issues':lineage_issues(records), 'deliberately_broken_example_issues':issues})
print('\n'.join(issues))


Duplicate ID: lab10-source-v1
Missing parent: missing-source for lab10-source-v1
Cycle at: lab10-summary-v1


## Package integrity and prepare a requirement review

Keep exact requirement text and clause references in a separately authorized review of the full standard. The concept rows below are local prompts, not IEEE clauses. Assign gaps and an owner; technical completeness must not turn pending publication into approved.


In [4]:
path = out/'provenance_catalog.json'
integrity = {'file':path.name, 'sha256':hashlib.sha256(path.read_bytes()).hexdigest(), 'scope':'Exact catalog bytes only; not authenticity, authority, or conformance'}
save_json('integrity_manifest.json', integrity)
review = [dict(local_concept=c, standard_clause='UNMAPPED: full-standard review required', applicability='undetermined', implementation=i, evidence=e, status='not assessed', owner='Designated standards reviewer', gap='Map applicable requirement and evaluate evidence') for c,i,e in [
    ('Identity and lineage','record_id, derived_from, processing_steps','provenance_catalog.json; lineage_review.json'),
    ('Governance and stewardship','Nation, steward, handling, pending release','provenance_catalog.json'),
    ('Integrity and lifecycle','SHA-256 catalog digest','integrity_manifest.json; lifecycle review still needed')]]
save_csv('standard_review_worksheet.csv', review)
assert all(r['status']=='not assessed' for r in review)
print('Toolkit schema exported; IEEE conformance not established.')


Toolkit schema exported; IEEE conformance not established.


## Independent task

Add a second source and a derived record with two parents. Explain which authority, use terms, and handling rules require reconciliation. Introduce one broken parent and repair it. If the full standard is available, map one applicable requirement with a real clause reference and evidence; otherwise leave it unassessed.

## Completion checklist

- [ ] Each parent resolves and IDs are unique.
- [ ] Deliberate missing-parent, duplicate, and cycle cases are detected.
- [ ] Integrity scope is explained.
- [ ] Requirement mapping remains unassessed until reviewed against the full standard.

## Primary references

- [IEEE 2890-2025 official standard page](https://standards.ieee.org/ieee/2890/10318/)